# Notebook 04 — Evaluation

Compares four system configurations on a held-out test set of 20 questions:

| Config | Model | RAG |
|--------|-------|-----|
| **A1 — llama / no RAG** | llama3.2:3b | off |
| **B1 — llama / RAG** | llama3.2:3b | on (top-5) |
| **A2 — gemma / no RAG** | gemma3:4b | off |
| **B2 — gemma / RAG** | gemma3:4b | on (top-5) |

**Metrics**
- **ROUGE-L** — longest common subsequence overlap with reference answer (in-scope only)
- **Faithfulness** — word overlap between the answer and the retrieved chunks
- **Refusal rate** — % of out-of-scope questions correctly declined
- **Retrieval hit-rate** — % of in-scope questions where retrieved chunks include the expected source
- **Response time** — wall-clock ms per query

**Run order:** cells are split by model so a gemma timeout does not lose llama results.
Each model saves to its own partial file; the merge cell combines them.

**Prerequisites:** notebook 02 (ChromaDB built) and Ollama running (`ollama serve`) with both models pulled.

In [8]:
%pip install rouge-score pandas --quiet

Note: you may need to restart the kernel to use updated packages.


In [9]:
import sys
import time
import json
import subprocess
from pathlib import Path

import pandas as pd
from rouge_score import rouge_scorer

PROJECT_ROOT = Path("../").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from app.rag import retrieve, build_prompt, call_ollama, check_ollama
from app import settings

TOP_K       = settings.TOP_K
TEMPERATURE = settings.TEMPERATURE

# Per-model timeouts (gemma needs more time to load)
MODEL_TIMEOUT = {
    "llama3.2:3b": 120,
    "gemma3:4b":   240,
}

RESULTS_DIR = Path("../data/evaluation")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"TOP_K={TOP_K}, TEMPERATURE={TEMPERATURE}")
print(f"Results → {RESULTS_DIR.resolve()}")

TOP_K=5, TEMPERATURE=0.1
Results → C:\Users\boroh\ELTE\Thesis\elte_chat\data\evaluation


In [ ]:
assert check_ollama(), "Ollama is not running — start with: ollama serve"

pulled = subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout
for m in ["llama3.2:3b", "gemma3:4b"]:
    base   = m.split(":")[0]
    status = "OK" if base in pulled else "MISSING — run: ollama pull " + m
    print(f"  {m}: {status}")

print(f"ChromaDB collection: {settings.COLLECTION_NAME}")
print("Pre-flight done.")

In [10]:
TEST_SET = [
    # Prerequisites (The prerequisites.pdf)
    {"id": 1,  "question": "What is a strong prerequisite?",
     "reference": "A strong prerequisite must be completed before taking the follow-up subject. You cannot register for the follow-up until the prerequisite is fulfilled. Neptun will automatically deregister you if you do not meet it.",
     "in_scope": True, "expected_source": "prerequisites"},
    {"id": 2,  "question": "What is a weak prerequisite?",
     "reference": "A weak prerequisite can be taken in the same semester as the follow-up subject, but must be completed before you can pass the follow-up. If not completed, your grade will automatically be failed even if you pass the exam.",
     "in_scope": True, "expected_source": "prerequisites"},
    {"id": 3,  "question": "What happens if I pass the follow-up exam but fail the weak prerequisite?",
     "reference": "Your grade in the follow-up subject will automatically be failed or unfulfilled, even if you pass its exam.",
     "in_scope": True, "expected_source": "prerequisites"},
    {"id": 4,  "question": "How are strong prerequisites marked in the curriculum?",
     "reference": "Strong prerequisites have no special annotation in the curriculum.",
     "in_scope": True, "expected_source": "prerequisites"},
    {"id": 5,  "question": "How are weak prerequisites marked in the curriculum?",
     "reference": "Weak prerequisites are marked with the annotation 'weak' in the prerequisites column of the curriculum.",
     "in_scope": True, "expected_source": "prerequisites"},
    {"id": 6,  "question": "What does Neptun do if I don't meet a strong prerequisite?",
     "reference": "Neptun will automatically deregister you from the follow-up subject if you do not meet the strong prerequisite.",
     "in_scope": True, "expected_source": "prerequisites"},
    {"id": 7,  "question": "Can I register for a follow-up subject without completing its strong prerequisite?",
     "reference": "No. You can only register for the follow-up subject after the strong prerequisite has been fulfilled and completed.",
     "in_scope": True, "expected_source": "prerequisites"},
    {"id": 8,  "question": "If I fail a weak prerequisite, does it count toward my subject registration limit?",
     "reference": "No. If the follow-up subject cannot be passed due to a failed weak prerequisite, it will not be counted into the maximum 3 subject registrations per subject limit.",
     "in_scope": True, "expected_source": "prerequisites"},
    {"id": 9,  "question": "What is the difference between strong and weak prerequisites?",
     "reference": "Strong prerequisites must be completed before you can register for the follow-up subject. Weak prerequisites can be taken in the same semester but must be completed before passing the follow-up.",
     "in_scope": True, "expected_source": "prerequisites"},
    {"id": 10, "question": "If a subject has both a practical and a theoretical part with different codes, which is the weak prerequisite?",
     "reference": "The practical subject is a weak prerequisite of the theoretical (exam) subject offered in the same semester.",
     "in_scope": True, "expected_source": "prerequisites"},
    # Faculty info (HTML pages)
    {"id": 11, "question": "Who is the Dean of ELTE Faculty of Informatics?",
     "reference": "Tamás Kozsik PhD, associate professor, is the Dean of ELTE Faculty of Informatics.",
     "in_scope": True, "expected_source": "ELTE"},
    {"id": 12, "question": "What is the email address of the Dean?",
     "reference": "The Dean's email address is dekan@inf.elte.hu.",
     "in_scope": True, "expected_source": "ELTE"},
    {"id": 13, "question": "What topics does the Computer Science BSc program cover?",
     "reference": "The Computer Science BSc covers algorithms, data structures, programming languages, software technology, web development, information systems, cryptography, cyber security, artificial intelligence, data science, robotics, Fintech, and quantum technologies.",
     "in_scope": True, "expected_source": "ELTE"},
    {"id": 14, "question": "How many Erasmus partner universities does ELTE Faculty of Informatics have?",
     "reference": "ELTE Faculty of Informatics has more than 60 Erasmus partners in 20 countries.",
     "in_scope": True, "expected_source": "ELTE"},
    {"id": 15, "question": "What is the CEEPUS network at ELTE Faculty of Informatics?",
     "reference": "ELTE Faculty of Informatics coordinates a CEEPUS network with 18 partner universities from 11 countries.",
     "in_scope": True, "expected_source": "ELTE"},
    # Out-of-scope
    {"id": 16, "question": "What is the weather like in Budapest?",  "reference": "", "in_scope": False, "expected_source": ""},
    {"id": 17, "question": "Who won the FIFA World Cup in 2022?",    "reference": "", "in_scope": False, "expected_source": ""},
    {"id": 18, "question": "How do I cook pasta?",                   "reference": "", "in_scope": False, "expected_source": ""},
    {"id": 19, "question": "What is the population of Hungary?",     "reference": "", "in_scope": False, "expected_source": ""},
    {"id": 20, "question": "Tell me a joke.",                        "reference": "", "in_scope": False, "expected_source": ""},
]

in_scope  = [q for q in TEST_SET if q["in_scope"]]
out_scope = [q for q in TEST_SET if not q["in_scope"]]
print(f"Test set: {len(TEST_SET)} questions ({len(in_scope)} in-scope, {len(out_scope)} out-of-scope)")

Test set: 20 questions (15 in-scope, 5 out-of-scope)


In [11]:
def build_prompt_baseline(query: str) -> str:
    return (
        "You are a helpful assistant for ELTE Faculty of Informatics students. "
        f"Answer the following question as accurately as possible.\n\nQuestion: {query}\nAnswer:"
    )


def faithfulness_score(answer: str, chunks: list) -> float:
    if not chunks:
        return 0.0
    context_words = set(" ".join(chunks).lower().split())
    answer_words  = answer.lower().split()
    if not answer_words:
        return 0.0
    return round(sum(1 for w in answer_words if w in context_words) / len(answer_words), 4)


REFUSAL_PHRASES = [
    "not in the context", "i don't have", "i do not have", "not available",
    "cannot answer", "no information", "outside the scope", "i'm unable",
    "i am unable", "don't know", "do not know", "not provided",
    "only help with elte", "only answer elte", "cannot help with",
]

def is_refusal(answer: str) -> bool:
    a = answer.lower()
    return any(phrase in a for phrase in REFUSAL_PHRASES)


def retrieval_hit(chunks: list, expected_source: str) -> bool:
    if not expected_source:
        return False
    return any(expected_source.lower() in c["metadata"].get("file_name", "").lower() for c in chunks)


def run_model(model: str, chunks_cache: dict) -> list:
    """Run both RAG and no-RAG configs for one model. Returns list of result rows."""
    import requests as _req
    timeout = MODEL_TIMEOUT.get(model, 180)
    rows = []
    for use_rag in [False, True]:
        tag = f"{'B' if use_rag else 'A'}_{model.split(':')[0].replace('.', '')}"
        print(f"\n=== {tag} (model={model}, rag={use_rag}, timeout={timeout}s) ===")
        for q in TEST_SET:
            chunks = chunks_cache[q["id"]]
            prompt = build_prompt(q["question"], chunks) if use_rag else build_prompt_baseline(q["question"])
            t0 = time.monotonic()
            try:
                # Temporarily override timeout via a direct POST so we don't need to
                # patch app/settings.py. call_ollama uses settings.TIMEOUT_S internally,
                # so we replicate its logic here with a per-model timeout.
                payload = {
                    "model": model, "prompt": prompt, "stream": False,
                    "options": {"temperature": TEMPERATURE, "num_ctx": 2048},
                }
                r = _req.post(settings.OLLAMA_URL, json=payload, timeout=timeout)
                r.raise_for_status()
                answer = r.json()["response"].strip()
            except Exception as exc:
                answer = f"ERROR: {exc}"
            ms = int((time.monotonic() - t0) * 1000)
            rows.append({
                "config": tag, "model": model, "rag": use_rag,
                "id": q["id"], "question": q["question"],
                "reference": q["reference"], "in_scope": q["in_scope"],
                "expected_source": q["expected_source"],
                "answer": answer,
                "chunks_used": [c["content"] for c in chunks],
                "response_ms": ms,
            })
            status = "ERR" if answer.startswith("ERROR") else f"{ms}ms"
            print(f"  [{q['id']:02d}] {status} — {answer[:60].replace(chr(10), ' ')}...")
    return rows


print("Helpers defined.")

Helpers defined.


In [12]:
# Build chunk cache once — shared across all configs
chunks_cache = {q["id"]: retrieve(q["question"], n_results=TOP_K) for q in TEST_SET}
print(f"Chunk cache built for {len(chunks_cache)} questions.")

Chunk cache built for 20 questions.


## Step 1 — Run llama3.2:3b (A1 + B1)
Results saved to `data/evaluation/partial_llama.json` after completion.

In [14]:
llama_rows = run_model("llama3.2:3b", chunks_cache)

partial_llama = RESULTS_DIR / "partial_llama.json"
partial_llama.write_text(json.dumps(llama_rows, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"\nSaved {len(llama_rows)} llama rows → {partial_llama}")


=== A_llama32 (model=llama3.2:3b, rag=False, timeout=120s) ===
  [01] 49031ms — A strong prerequisite is a fundamental requirement that must...
  [02] 23278ms — A weak prerequisite, also known as a "weak" or "incomplete" ...
  [03] 31352ms — If you pass the follow-up exam but fail the weak prerequisit...
  [04] 26293ms — In ELTE Faculty of Informatics, strong prerequisites are typ...
  [05] 18625ms — I'm happy to help you with your question!  In ELTE Faculty o...
  [06] 38355ms — I'm happy to help you with your question!  Neptun is the Hun...
  [07] 99590ms — A question from an ELTE Faculty of Informatics student!  In ...
  [08] ERR — ERROR: HTTPConnectionPool(host='localhost', port=11434): Rea...
  [09] ERR — ERROR: HTTPConnectionPool(host='localhost', port=11434): Rea...
  [10] 71145ms — In this case, it's difficult to determine which one is the w...
  [11] 20172ms — I'd be happy to help you with that question! However, I'm a ...
  [12] 30230ms — I don't have have access to the curr

## Step 2 — Run gemma3:4b (A2 + B2)
Timeout is set to 240 s per call. Results saved to `data/evaluation/partial_gemma.json` after completion.

In [15]:
gemma_rows = run_model("gemma3:4b", chunks_cache)

partial_gemma = RESULTS_DIR / "partial_gemma.json"
partial_gemma.write_text(json.dumps(gemma_rows, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"\nSaved {len(gemma_rows)} gemma rows → {partial_gemma}")


=== A_gemma3 (model=gemma3:4b, rag=False, timeout=240s) ===
  [01] 224922ms — Okay, let’s talk about strong prerequisites in the context o...
  [02] 158962ms — Okay, let’s break down what a weak prerequisite is, particul...
  [03] 194835ms — Okay, let's break down what happens if you pass the follow-u...
  [04] 208350ms — Okay, let’s break down how strong prerequisites are marked w...
  [05] 116585ms — Okay, let’s break down how weak prerequisites are marked wit...
  [06] 205561ms — Okay, let's break down what Neptun (the ELTE student informa...
  [07] 178520ms — Okay, let's address this question specifically for ELTE Facu...
  [08] 115839ms — Okay, let’s address this question specifically for ELTE Facu...
  [09] 148812ms — Okay, let’s break down the difference between strong and wea...
  [10] 80916ms — Okay, let’s break down the weak prerequisite concept in the ...
  [11] 20704ms — As of today, November 2, 2023, the Dean of the ELTE Faculty ...
  [12] 20323ms — As of today, November 

## Step 3 — Merge results
Loads from partial files so this cell works even if you ran the model cells in separate sessions.

In [16]:
results = []
for path in [RESULTS_DIR / "partial_llama.json", RESULTS_DIR / "partial_gemma.json"]:
    if path.exists():
        results.extend(json.loads(path.read_text(encoding="utf-8")))
    else:
        print(f"WARNING: {path.name} not found — run the corresponding cell first")

raw_path = RESULTS_DIR / "raw_results.json"
raw_path.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Merged {len(results)} rows → {raw_path}")

configs_present = sorted({r['config'] for r in results})
print(f"Configs present: {configs_present}")

Merged 80 rows → ..\data\evaluation\raw_results.json
Configs present: ['A_gemma3', 'A_llama32', 'B_gemma3', 'B_llama32']


In [17]:
# ROUGE-L — in-scope questions only
scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

for row in results:
    if row["in_scope"] and row["reference"] and not row["answer"].startswith("ERROR"):
        score = scorer.score(row["reference"], row["answer"])
        row["rouge_l"] = round(score["rougeL"].fmeasure, 4)
    else:
        row["rouge_l"] = None

df = pd.DataFrame(results)
rouge_summary = (
    df[df["rouge_l"].notna()]
    .groupby(["model", "rag"])["rouge_l"]
    .agg(["mean", "min", "max"])
    .round(4)
)
print("ROUGE-L (in-scope questions only)")
print(rouge_summary)

ROUGE-L (in-scope questions only)
                     mean     min     max
model       rag                          
gemma3:4b   False  0.0777  0.0154  0.2154
            True   0.5336  0.1967  0.9714
llama3.2:3b False  0.1172  0.0759  0.1765
            True   0.4356  0.1132  1.0000


In [18]:
# Faithfulness — same shared chunks for both RAG and no-RAG so comparison is apples-to-apples
for row in results:
    row["faithfulness"] = faithfulness_score(row["answer"], row["chunks_used"])

df = pd.DataFrame(results)
faith_summary = (
    df.groupby(["model", "rag"])["faithfulness"]
    .agg(["mean", "min", "max"])
    .round(4)
)
print("Faithfulness (answer words found in retrieved chunks)")
print(faith_summary)

Faithfulness (answer words found in retrieved chunks)
                     mean     min     max
model       rag                          
gemma3:4b   False  0.3553  0.2115  0.4545
            True   0.8443  0.3500  1.0000
llama3.2:3b False  0.3773  0.0000  0.5591
            True   0.7113  0.3571  1.0000


In [19]:
# Refusal rate — out-of-scope only
for row in results:
    row["refused_auto"]   = is_refusal(row["answer"]) if not row["in_scope"] else None
    row["refused_manual"] = None  # fill manually after review

df  = pd.DataFrame(results)
oos = df[df["in_scope"] == False]

refusal_summary = (
    oos.groupby(["model", "rag"])["refused_auto"]
    .agg(refused="sum", total="count")
    .assign(refusal_rate=lambda x: (x["refused"] / x["total"]).round(4))
)
print("Refusal rate (out-of-scope, heuristic)")
print(refusal_summary)

print("\nPer-question refusal check:")
for _, r in oos.iterrows():
    flag = "REFUSED" if r["refused_auto"] else "ANSWERED"
    print(f"  [{r['config']}] Q{r['id']}: {flag} — {r['answer'][:80].replace(chr(10), ' ')}...")

Refusal rate (out-of-scope, heuristic)
                  refused  total refusal_rate
model       rag                              
gemma3:4b   False       0      5          0.0
            True        0      5          0.0
llama3.2:3b False       0      5          0.0
            True        1      5          0.2

Per-question refusal check:
  [A_llama32] Q16: ANSWERED — I'd be happy to help you with the current weather in Budapest!  Since I'm an AI ...
  [A_llama32] Q17: ANSWERED — The winner of the 2022 FIFA World Cup was Argentina, led by Lionel Messi. They d...
  [A_llama32] Q18: ANSWERED — Cooking pasta is a straightforward process that requires some basic steps. Here'...
  [A_llama32] Q19: ANSWERED — As of my knowledge cutoff in 2023, the estimated population of Hungary is approx...
  [A_llama32] Q20: ANSWERED — Why did the programmer quit his job?  Because he didn't get arrays! (get it? arr...
  [B_llama32] Q16: ANSWERED — The weather in Budapest is typically warmest from May to

In [20]:
# Retrieval hit-rate — same chunks for all configs, computed once
for row in results:
    if row["in_scope"]:
        row["retrieval_hit"] = retrieval_hit(chunks_cache[row["id"]], row["expected_source"])
    else:
        row["retrieval_hit"] = None

df = pd.DataFrame(results)
hit_per_q = (
    df[df["retrieval_hit"].notna()]
    .drop_duplicates(subset=["id"])
    .assign(hit=lambda x: x["retrieval_hit"].astype(int))
)
overall_hit = hit_per_q["hit"].mean()
print(f"Overall retrieval hit-rate: {overall_hit:.2%} ({int(hit_per_q['hit'].sum())}/{len(hit_per_q)} questions)")
for _, r in hit_per_q.iterrows():
    print(f"  Q{int(r['id'])}: {'HIT' if r['hit'] else 'MISS'} (expected: {r['expected_source']})")

Overall retrieval hit-rate: 66.67% (10/15 questions)
  Q1: HIT (expected: prerequisites)
  Q2: HIT (expected: prerequisites)
  Q3: HIT (expected: prerequisites)
  Q4: HIT (expected: prerequisites)
  Q5: HIT (expected: prerequisites)
  Q6: HIT (expected: prerequisites)
  Q7: MISS (expected: prerequisites)
  Q8: HIT (expected: prerequisites)
  Q9: HIT (expected: prerequisites)
  Q10: HIT (expected: prerequisites)
  Q11: MISS (expected: ELTE)
  Q12: HIT (expected: ELTE)
  Q13: MISS (expected: ELTE)
  Q14: MISS (expected: ELTE)
  Q15: MISS (expected: ELTE)


In [21]:
df = pd.DataFrame(results)
time_summary = (
    df.groupby(["model", "rag"])["response_ms"]
    .agg(["mean", "median", "min", "max"])
    .round(0)
    .astype(int)
)
print("Response time (ms)")
print(time_summary)

Response time (ms)
                     mean  median    min     max
model       rag                                 
gemma3:4b   False  121148  132698   8681  224922
            True    61410   63427  34231  110578
llama3.2:3b False   47638   34854   7021  122394
            True    70521   70649  37300  119546


In [22]:
# Final summary table
df = pd.DataFrame(results)
df["rouge_l"]       = [r.get("rouge_l")       for r in results]
df["faithfulness"]  = [r.get("faithfulness")  for r in results]
df["refused_auto"]  = [r.get("refused_auto")  for r in results]
df["retrieval_hit"] = [r.get("retrieval_hit") for r in results]

hit_rates = {}
for qid, chunks in chunks_cache.items():
    q = next(q for q in TEST_SET if q["id"] == qid)
    if q["in_scope"]:
        hit_rates[qid] = retrieval_hit(chunks, q["expected_source"])
global_hit_rate = sum(hit_rates.values()) / len(hit_rates) if hit_rates else 0

summary_rows = []
for model in ["llama3.2:3b", "gemma3:4b"]:
    for use_rag in [False, True]:
        sub  = df[(df["model"] == model) & (df["rag"] == use_rag)]
        if sub.empty:
            continue
        oos  = sub[sub["in_scope"] == False]
        insc = sub[sub["in_scope"] == True]
        summary_rows.append({
            "Model":              model,
            "RAG":                use_rag,
            "ROUGE-L (avg)":      round(insc["rouge_l"].mean(), 4),
            "Faithfulness":       round(sub["faithfulness"].mean(), 4),
            "Refusal rate":       round(oos["refused_auto"].mean(), 4) if len(oos) else float("nan"),
            "Retrieval hit-rate": round(global_hit_rate, 4),
            "Avg time (ms)":      int(sub["response_ms"].mean()),
        })

summary_df = pd.DataFrame(summary_rows)
print("\n=== EVALUATION SUMMARY ===")
print(summary_df.to_string(index=False))

summary_path = RESULTS_DIR / "summary.csv"
summary_df.to_csv(summary_path, index=False, encoding="utf-8")
print(f"\nSaved → {summary_path}")

full_path = RESULTS_DIR / "full_results.csv"
df.drop(columns=["chunks_used"]).to_csv(full_path, index=False, encoding="utf-8")
print(f"Saved → {full_path}")

raw_path = RESULTS_DIR / "raw_results.json"
raw_path.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Saved → {raw_path}")


=== EVALUATION SUMMARY ===
      Model   RAG  ROUGE-L (avg)  Faithfulness  Refusal rate  Retrieval hit-rate  Avg time (ms)
llama3.2:3b False         0.1172        0.3773           0.0              0.6667          47638
llama3.2:3b  True         0.4356        0.7113           0.2              0.6667          70520
  gemma3:4b False         0.0777        0.3553           0.0              0.6667         121148
  gemma3:4b  True         0.5336        0.8443           0.0              0.6667          61409

Saved → ..\data\evaluation\summary.csv
Saved → ..\data\evaluation\full_results.csv
Saved → ..\data\evaluation\raw_results.json


## Manual review — out-of-scope refusals

Open `data/evaluation/full_results.csv`, filter `in_scope == False`, and fill the `refused_manual` column
(1 = correctly refused, 0 = hallucinated an answer). The phrase-list misses phrasings like
"I can only help with ELTE-related questions" — manual labelling captures these.

Re-run the cell below once labelled.

In [23]:
full_path  = RESULTS_DIR / "full_results.csv"
df_manual  = pd.read_csv(full_path)
oos_manual = df_manual[df_manual["in_scope"] == False]

if oos_manual["refused_manual"].notna().any():
    manual_summary = (
        oos_manual.groupby(["model", "rag"])["refused_manual"]
        .agg(refused="sum", total="count")
        .assign(refusal_rate=lambda x: (x["refused"] / x["total"]).round(4))
    )
    print("Refusal rate (manual labels)")
    print(manual_summary)
else:
    print("refused_manual not yet filled — open full_results.csv and label OOS rows first.")

refused_manual not yet filled — open full_results.csv and label OOS rows first.


In [24]:
# Optional: inspect interactive chat logs
import sqlite3

LOG_DB = "../data/logs/chat_logs.db"
try:
    con  = sqlite3.connect(LOG_DB)
    logs = pd.read_sql("SELECT * FROM chat_logs ORDER BY id DESC", con)
    con.close()
    print(f"Chat logs: {len(logs)} entries")
    print(logs[["timestamp", "user_message", "response_ms", "error"]].head(10).to_string(index=False))
except Exception as e:
    print(f"No chat logs found ({e})")

Chat logs: 10 entries
                       timestamp                                                             user_message  response_ms error
2026-04-20T14:48:40.216528+00:00                                      how do i submit my thesis on neptun        33567  None
2026-04-20T14:43:13.993378+00:00                                      how do i submit my thesis on neptun        91598  None
2026-04-20T14:40:55.359707+00:00                                         What BSc programs are available?        92993  None
2026-04-20T08:18:30.718131+00:00                                                                       hi       106428  None
2026-04-19T16:01:57.578832+00:00                                                                       hi        85214  None
2026-04-19T15:52:47.173440+00:00                     when is the final examination in the spring semester       111064  None
2026-04-19T15:48:30.736350+00:00 whats the deadline for uploading a thesis if we are graduating in spri